In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 0. File paths — neutral only, update timestamps
json_files = [
    "doubao_matrix_neutral_dilemma_3_280_20260411_160225.json",
    "gemini_matrix_neutral_dilemma_3_280_20260410_185748.json",
    "openai_matrix_neutral_dilemma_3_280_20260411_210448.json",
]

def infer_model(path):
    name = os.path.basename(path).lower()
    for m in ("doubao", "gemini", "openai"):
        if m in name:
            return m
    return "unknown"

# 1. Load
frames = []
for fp in json_files:
    with open(fp, "r", encoding="utf-8") as f:
        data = json.load(f)
    tmp = pd.DataFrame(data if isinstance(data, list) else [data])
    tmp["model_family"] = infer_model(fp)
    frames.append(tmp)

df_raw = pd.concat(frames, ignore_index=True)
print("Loaded rows:", len(df_raw))

# 2. Parse decision
def extract_decision(text):
    if not isinstance(text, str):
        return np.nan
    m = re.search(r"DECISION:\s*(SWITCH|NO_SWITCH|NO SWITCH|STAY)", text.upper())
    if not m:
        return np.nan
    d = m.group(1).replace(" ", "_")
    return "NO_SWITCH" if d == "STAY" else d

df = df_raw.copy()
df["decision"]     = df["content"].apply(extract_decision)
df["decision_val"] = df["decision"].map({"SWITCH": 1, "NO_SWITCH": 0})
df = df[df["decision_val"].notna()].copy()
df["decision_val"] = df["decision_val"].astype(int)

print("\nDecision counts:")
print(df["decision"].value_counts(dropna=False))

ROLE_ORDER = ["role_A", "role_B", "role_C", "role_D", "role_E", "role_F", "role_G"]

# 3. Bootstrap: 95% CI for mean(x) - mean(y)
def bootstrap_diff_ci(x, y, n_boot=500, seed=42):
    rng = np.random.default_rng(seed)
    x, y = np.asarray(x, float), np.asarray(y, float)
    diffs = [
        rng.choice(x, size=len(x), replace=True).mean() -
        rng.choice(y, size=len(y), replace=True).mean()
        for _ in range(n_boot)
    ]
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    return x.mean() - y.mean(), ci_low, ci_high

# 4. Baseline = role_A per model; compare each role
baseline_pool = (
    df[df["role"] == "role_A"]
    .groupby("model_family")["decision_val"]
    .apply(list).to_dict()
)

rows = []
for (model, role), grp in df.groupby(["model_family", "role"]):
    x = grp["decision_val"].values
    y = np.asarray(baseline_pool.get(model, []), float)
    if len(x) == 0 or len(y) == 0:
        rows.append(dict(model_family=model, role=role, n=len(x),
                         p_switch=np.nan, diff=np.nan, ci_low=np.nan,
                         ci_high=np.nan, sig=False, direction="—"))
        continue
    diff, ci_low, ci_high = bootstrap_diff_ci(x, y)
    sig = (ci_high < 0) or (ci_low > 0)
    rows.append(dict(
        model_family=model, role=role, n=int(len(x)),
        p_switch=round(float(x.mean()), 3),
        diff=round(diff, 3),
        ci_low=round(ci_low, 3),
        ci_high=round(ci_high, 3),
        sig=sig,
        direction=("↑" if diff > 0 else "↓") if sig else "ns",
    ))

result = pd.DataFrame(rows)
result["role"] = pd.Categorical(result["role"], categories=ROLE_ORDER, ordered=True)
result = result.sort_values(["model_family", "role"]).reset_index(drop=True)

# 5. Display
print("\n=== Bootstrap ΔP(SWITCH) vs Baseline (role_A, same model) ===\n")
display(result)

pivot_diff = result.pivot_table(index="role", columns="model_family", values="diff")
pivot_sig  = result.pivot_table(index="role", columns="model_family", values="direction", aggfunc="first")
print("\n--- ΔP pivot ---")
display(pivot_diff.round(3))
print("\n--- Significance (↑/↓/ns) pivot ---")
display(pivot_sig)

result.to_csv("bootstrap_diff_vs_baseline.csv", index=False, encoding="utf-8-sig")
print("Saved: bootstrap_diff_vs_baseline.csv")

# 6. Bar plot
ROLE_LABELS = {r: r.replace("role_", "") for r in ROLE_ORDER}
role_plot   = [r for r in ROLE_ORDER if r != "role_A"]
models      = sorted(result["model_family"].dropna().unique())
colors      = {"doubao": "#8172B3", "gemini": "#4C72B0", "openai": "#DD8452"}

fig, axes = plt.subplots(1, len(models), figsize=(18, 5), sharey=True)
if len(models) == 1:
    axes = [axes]

for ax, model in zip(axes, models):
    sub  = result[result["model_family"] == model].set_index("role").reindex(role_plot).reset_index()
    xpos = np.arange(len(role_plot))
    y    = sub["diff"].values
    yerr = np.vstack([y - sub["ci_low"].values, sub["ci_high"].values - y])
    ax.bar(xpos, y, width=0.5, alpha=0.85, color=colors.get(model, "#888888"))
    ax.errorbar(xpos, y, yerr=np.abs(yerr), fmt="none", capsize=4, linewidth=1.2, color="black")
    for j, (yv, s) in enumerate(zip(y, sub["sig"].values)):
        if s:
            ax.text(xpos[j], yv + (0.02 if yv >= 0 else -0.05), "*",
                    ha="center", va="bottom", fontsize=13, color="black")
    ax.axhline(0, linestyle="--", linewidth=1.2, color="gray")
    ax.set_xticks(xpos)
    ax.set_xticklabels([ROLE_LABELS[r] for r in role_plot])
    ax.set_xlabel("Role")
    ax.set_title(f"Model: {model}")

axes[0].set_ylabel("ΔP(SWITCH) vs Baseline (role_A)")
fig.suptitle("Role Effect on SWITCH Probability\n(* = 95% bootstrap CI excludes 0)", fontsize=13)
plt.tight_layout()
plt.savefig("bootstrap_diff_vs_baseline.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: bootstrap_diff_vs_baseline.png")


In [ ]:
import seaborn as sns

N_BOOT = 500
SEED   = 43
rng    = np.random.default_rng(SEED)

ROLE_ORDER   = ["role_A","role_B","role_C","role_D","role_E","role_F","role_G"]
ROLE_LABEL   = {r: f"Role {r[-1].upper()}" for r in ROLE_ORDER}
MODEL_TITLES = {"doubao": "Doubao", "gemini": "Gemini-2.5-flash", "openai": "gpt-4o"}

# 1. Bootstrap diffs per (model, role), baseline = role_A
boot_records = []
for model, mdf in df.groupby("model_family"):
    base_vals = mdf[mdf["role"] == "role_A"]["decision_val"].values
    for role, grp in mdf.groupby("role"):
        x = grp["decision_val"].values
        if len(x) == 0 or len(base_vals) == 0:
            continue
        for _ in range(N_BOOT):
            xb = rng.choice(x,         size=len(x),         replace=True).mean()
            yb = rng.choice(base_vals, size=len(base_vals), replace=True).mean()
            boot_records.append({"model": model, "role": role, "boot_diff": float(xb - yb)})

boot_df = pd.DataFrame(boot_records)

# 2. Significance
sig_map = {}
for (model, role), grp in boot_df.groupby(["model", "role"]):
    lo, hi = np.percentile(grp["boot_diff"].values, [2.5, 97.5])
    sig_map[(model, role)] = (lo > 0) or (hi < 0)

# 3. Color per role
palette    = sns.color_palette("Blues_d", len(ROLE_ORDER))
role_color = {role: palette[i] for i, role in enumerate(ROLE_ORDER)}
boot_df["x_idx"] = boot_df["role"].map({r: i for i, r in enumerate(ROLE_ORDER)})

# 4. Plot — one figure per model
MODELS = sorted(df["model_family"].dropna().unique())

for model in MODELS:
    sub = boot_df[boot_df["model"] == model].sort_values("x_idx")

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    for xi, role in enumerate(ROLE_ORDER):
        cell   = sub[sub["role"] == role]["boot_diff"].values
        if len(cell) == 0:
            continue
        is_sig = sig_map.get((model, role), False)
        color  = role_color[role]

        parts = ax.violinplot(cell, positions=[xi], widths=0.65,
                              showmedians=False, showextrema=False)
        for pc in parts["bodies"]:
            pc.set_facecolor(color)
            pc.set_edgecolor("#333333" if is_sig else "#999999")
            pc.set_alpha(0.85 if is_sig else 0.5)
            pc.set_linewidth(1.2 if is_sig else 0.4)

        q25, med, q75 = np.percentile(cell, [25, 50, 75])
        ax.plot([xi, xi], [q25, q75], color="#111111" if is_sig else "#888888",
                lw=3.5 if is_sig else 2.0, solid_capstyle="round", zorder=3)
        ax.plot(xi, med, "o", color="white",
                markeredgecolor="#111111" if is_sig else "#888888",
                markersize=6, markeredgewidth=1.2, zorder=4)
        if is_sig:
            ax.text(xi, max(cell) + 0.01, "*", ha="center", va="bottom",
                    fontsize=13, color="#c0392b", fontweight="bold", zorder=5)

    ax.axhline(0, color="#555555", linewidth=1.0, linestyle="--", alpha=0.7)
    ax.set_xticks(range(len(ROLE_ORDER)))
    ax.set_xticklabels([ROLE_LABEL[r] for r in ROLE_ORDER], fontsize=10)
    ax.set_ylabel("Bootstrap ΔP(SWITCH)\nvs Baseline (role_A)", fontsize=11)
    ax.set_title(
        f"SWITCH Probability Δ from Baseline — {MODEL_TITLES.get(model, model)}\n"
        f"({N_BOOT} bootstrap iterations, * = 95% CI excludes 0)",
        fontsize=13, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, color="#dddddd", linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_xlim(-0.6, len(ROLE_ORDER) - 0.4)

    plt.tight_layout()
    out = f"violin_switch_diff_{model}.png"
    plt.savefig(out, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {out}")
